import os, warnings, torch

import numpy as np
import scanpy as sc
import pandas as pd

from datasets.data_manager_SMA import SMA
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from collections import defaultdict

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


In [ ]:
import os, warnings, torch

import numpy as np
import scanpy as sc
import pandas as pd

from datasets.data_manager_SMA import SMA
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from collections import defaultdict

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


### Load dataset

In [ ]:
# please load the target Visium data (from the uploaded)
adata_path = '/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used/V11L12-109_A1'
rna_adata = sc.read_visium(path=adata_path, count_file='filtered_feature_bc_matrix.h5')

In [ ]:
coordiantes = []
for i in range(rna_adata.shape[0]):
    coordiantes.append(str(rna_adata.obs['array_row'].values[i]) + '_' + str(rna_adata.obs['array_col'].values[i]) )
coordiantes = np.array(coordiantes)

rna_adata.obs['coordinates'] = coordiantes

### Load args

In [ ]:
%run ./args/args_SMA.py
args = args

### Create dataloader

In [ ]:
# create the dataset
dataset = SMA(
    path_img=args.path_img,
    rna_path=args.rna_path,
    msi_path=args.msi_path,
    n_top_genes=args.n_source,
    n_top_targets=args.n_target,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Model initialization

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=False).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_SMA_last.pth', device=device)


### Model inference 

In [ ]:
pd_dictionary, gt_dictionary = defaultdict(), defaultdict()
pd_value, gt_value, node_ids = collect_predictions(
    model,
    dataset.testing,
    split='test',
    device=device,
    clip_min=0.0,
)

for node_id, pred, target in zip(node_ids, pd_value.numpy(), gt_value.numpy()):
    _, coordinates = node_id.split('/', 1)
    pd_dictionary[coordinates] = pred
    gt_dictionary[coordinates] = target

df_pd = pd.DataFrame.from_dict(pd_dictionary, orient='index', columns=['pd_' + str(i) for i in dataset.target_panel])
df_gt = pd.DataFrame.from_dict(gt_dictionary, orient='index', columns=['gt_' + str(i) for i in dataset.target_panel])


### Model evaluation

In [ ]:
from utils.evaluation import evaluator
pearson_sample_list, spearman_sample_list, _ = evaluator(df_pd.values, df_gt.values)

print(pearson_sample_list.mean())
print(spearman_sample_list.mean())

### Save the predictions to adata

In [ ]:
rna_adata.obs = rna_adata.obs.reset_index().merge(df_gt, left_on='coordinates', right_index=True, how='left').set_index('index')
rna_adata.obs = rna_adata.obs.reset_index().merge(df_pd, left_on='coordinates', right_index=True, how='left').set_index('index')

rna_adata = rna_adata[~np.isnan(rna_adata.obs['gt_390.16864'].values)] 

### Visualization

In [ ]:
index = 9
name = dataset.target_panel[index]

key_pd, key_gt = 'pd_' + name, 'gt_' + name

temp_array_gt = rna_adata.obs[key_gt].values.copy()
temp_array_pd = rna_adata.obs[key_pd].values.copy()

temp_array_pd = np.exp(temp_array_pd) -1
temp_array_gt =  np.exp(temp_array_gt) -1


# min_value = np.nanmin(temp_array_pd)
# filter out the abnormal spots
target_value = np.nanmean(temp_array_gt) - 8 * np.nanstd(temp_array_gt) 
temp_mask = temp_array_gt < target_value
temp_array_gt[temp_mask] = None

key_norm = 'norm_' + name
rna_adata.obs[key_norm] = (temp_array_gt - np.nanmin(temp_array_gt)) / (np.nanmax(temp_array_gt) - np.nanmin(temp_array_gt))

pd_norm_name = 'pd_norm_' + name
rna_adata.obs[pd_norm_name] = (temp_array_pd - np.nanmin(temp_array_pd)) / (np.nanmax(temp_array_pd) - np.nanmin(temp_array_pd))

################
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
sc.pl.embedding(rna_adata, basis='spatial', color=key_norm, title=f'Ground Truth {name}', ax=axs[0], show=False, cmap=Tropic_7.mpl_colormap) 
sc.pl.embedding(rna_adata, basis='spatial', color=pd_norm_name, title=f'Prediction {name}', ax=axs[1], show=False, cmap=Tropic_7.mpl_colormap) 

In [ ]:
index = 25
name = dataset.target_panel[index]

key_pd, key_gt = 'pd_' + name, 'gt_' + name

temp_array_gt = rna_adata.obs[key_gt].values.copy()
temp_array_pd = rna_adata.obs[key_pd].values.copy()

temp_array_pd = np.exp(temp_array_pd) -1
temp_array_gt =  np.exp(temp_array_gt) -1


# min_value = np.nanmin(temp_array_pd)
# filter out the abnormal spots
target_value = np.nanmean(temp_array_gt) - 8 * np.nanstd(temp_array_gt) 
temp_mask = temp_array_gt < target_value
temp_array_gt[temp_mask] = None

key_norm = 'norm_' + name
rna_adata.obs[key_norm] = (temp_array_gt - np.nanmin(temp_array_gt)) / (np.nanmax(temp_array_gt) - np.nanmin(temp_array_gt))

pd_norm_name = 'pd_norm_' + name
rna_adata.obs[pd_norm_name] = (temp_array_pd - np.nanmin(temp_array_pd)) / (np.nanmax(temp_array_pd) - np.nanmin(temp_array_pd))

################
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
sc.pl.embedding(rna_adata, basis='spatial', color=key_norm, title=f'Ground Truth {name}', ax=axs[0], show=False, cmap=Tropic_7.mpl_colormap) 
sc.pl.embedding(rna_adata, basis='spatial', color=pd_norm_name, title=f'Prediction {name}', ax=axs[1], show=False, cmap=Tropic_7.mpl_colormap) 

### Generate new adata for potential analysis, e.g., multi-omics domain identification

In [ ]:
gt_list = defaultdict(list)
pd_list = defaultdict(list)

for index, name in enumerate(rna_adata.obs.columns.tolist()):
    if name[0:2] == 'gt':
        gt_list[name.split('_')[-1]] = rna_adata.obs[name].values
    elif name[0:2] == 'pd':
        pd_list[name.split('_')[-1]] = rna_adata.obs[name].values

gt_dataframe = pd.DataFrame(gt_list)
pd_dataframe = pd.DataFrame(pd_list)

gt_adata = sc.AnnData(gt_dataframe)
pd_adata = sc.AnnData(pd_dataframe)

gt_adata.obsm['spatial'] = rna_adata.obsm['spatial']
pd_adata.obsm['spatial'] = rna_adata.obsm['spatial']

gt_adata.obs.index = rna_adata.obs.index.values
pd_adata.obs.index = rna_adata.obs.index.values